# Análisis de Evaluación de Sistemas de Recomendación

Este notebook analiza los resultados de la evaluación offline de diferentes estrategias de recomendación usando múltiples métricas.

## Métricas evaluadas:
- **NDCG@9**: Normalized Discounted Cumulative Gain
- **Precision@9**: Proporción de recomendaciones relevantes
- **Recall@9**: Proporción de items relevantes recuperados
- **F1@9**: Media armónica de Precision y Recall
- **MRR**: Mean Reciprocal Rank (posición del primer item relevante)
- **Genre Diversity**: Diversidad de géneros en las recomendaciones
- **Artist Diversity**: Diversidad de artistas en las recomendaciones
- **Novelty**: Novedad basada en popularidad (-log2)

## Estrategias evaluadas:
- **hybrid**: Sistema híbrido completo
- **advanced**: Recomendaciones avanzadas (NMF + Two Towers)
- **nmf**: Factorización matricial (NMF)
- **two_towers**: Two Towers (Deep Learning)
- **pairs**: Co-ocurrencia (release_pairs)
- **content**: Perfiles de contenido
- **random**: Exploración aleatoria (baseline)
- **popular**: Popularidad (baseline)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


# Configurar estilo
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["font.size"] = 11

# Cargar datos
csv_path = Path("../offline_recommender/output/resultados_evaluacion_completa.csv")
df = pd.read_csv(csv_path)

print(f"📊 Datos cargados: {len(df)} usuarios evaluados")
print(f"📈 Columnas: {len(df.columns)}")
print("\nPrimeras filas:")
df.head()

In [ ]:
# Definir estrategias y métricas
strategies = ["hybrid", "advanced", "nmf", "two_towers", "pairs", "content", "random", "popular"]
metrics = [
    "ndcg",
    "precision",
    "recall",
    "f1",
    "mrr",
    "genre_diversity",
    "artist_diversity",
    "novelty",
]

# Calcular estadísticas descriptivas por estrategia y métrica
print("=" * 80)
print("ESTADÍSTICAS DESCRIPTIVAS POR ESTRATEGIA")
print("=" * 80)

for strategy in strategies:
    print(f"\n{strategy.upper()}:")
    print("-" * 40)
    for metric in metrics:
        col = f"{strategy}_{metric}"
        if col in df.columns:
            values = df[col].dropna()
            if len(values) > 0:
                print(
                    f"  {metric:20s}: mean={values.mean():.4f}  std={values.std():.4f}  min={values.min():.4f}  max={values.max():.4f}"
                )

In [ ]:
# Crear DataFrame con promedios por estrategia
summary_data = []
for strategy in strategies:
    row = {"strategy": strategy}
    for metric in metrics:
        col = f"{strategy}_{metric}"
        if col in df.columns:
            row[metric] = df[col].mean()
        else:
            row[metric] = 0.0
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.set_index("strategy")

print("=" * 80)
print("PROMEDIOS POR ESTRATEGIA")
print("=" * 80)
print(summary_df.round(4))

In [ ]:
# Visualización 1: Comparación de métricas principales
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Comparación de Estrategias por Métricas Principales", fontsize=16, fontweight="bold")

# NDCG
ax1 = axes[0, 0]
summary_df["ndcg"].plot(kind="bar", ax=ax1, color="steelblue")
ax1.set_title("NDCG@9", fontweight="bold")
ax1.set_ylabel("NDCG")
ax1.set_xlabel("Estrategia")
ax1.tick_params(axis="x", rotation=45)
ax1.grid(axis="y", alpha=0.3)

# Precision
ax2 = axes[0, 1]
summary_df["precision"].plot(kind="bar", ax=ax2, color="coral")
ax2.set_title("Precision@9", fontweight="bold")
ax2.set_ylabel("Precision")
ax2.set_xlabel("Estrategia")
ax2.tick_params(axis="x", rotation=45)
ax2.grid(axis="y", alpha=0.3)

# Recall
ax3 = axes[1, 0]
summary_df["recall"].plot(kind="bar", ax=ax3, color="mediumseagreen")
ax3.set_title("Recall@9", fontweight="bold")
ax3.set_ylabel("Recall")
ax3.set_xlabel("Estrategia")
ax3.tick_params(axis="x", rotation=45)
ax3.grid(axis="y", alpha=0.3)

# F1
ax4 = axes[1, 1]
summary_df["f1"].plot(kind="bar", ax=ax4, color="mediumpurple")
ax4.set_title("F1@9", fontweight="bold")
ax4.set_ylabel("F1 Score")
ax4.set_xlabel("Estrategia")
ax4.tick_params(axis="x", rotation=45)
ax4.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualización 2: Heatmap de todas las métricas
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    summary_df.T,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    cbar_kws={"label": "Valor promedio"},
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Heatmap de Métricas por Estrategia", fontsize=14, fontweight="bold", pad=20)
ax.set_xlabel("Estrategia", fontweight="bold")
ax.set_ylabel("Métrica", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 3: Diversidad y Novedad
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Análisis de Diversidad y Novedad", fontsize=16, fontweight="bold")

# Genre Diversity
ax1 = axes[0]
summary_df["genre_diversity"].plot(kind="bar", ax=ax1, color="teal")
ax1.set_title("Diversidad de Géneros", fontweight="bold")
ax1.set_ylabel("Géneros únicos / Releases")
ax1.set_xlabel("Estrategia")
ax1.tick_params(axis="x", rotation=45)
ax1.grid(axis="y", alpha=0.3)

# Artist Diversity
ax2 = axes[1]
summary_df["artist_diversity"].plot(kind="bar", ax=ax2, color="orange")
ax2.set_title("Diversidad de Artistas", fontweight="bold")
ax2.set_ylabel("Artistas únicos / Releases")
ax2.set_xlabel("Estrategia")
ax2.tick_params(axis="x", rotation=45)
ax2.grid(axis="y", alpha=0.3)

# Novelty
ax3 = axes[2]
summary_df["novelty"].plot(kind="bar", ax=ax3, color="crimson")
ax3.set_title("Novedad", fontweight="bold")
ax3.set_ylabel("Novedad promedio (-log2)")
ax3.set_xlabel("Estrategia")
ax3.tick_params(axis="x", rotation=45)
ax3.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualización 4: Boxplots de distribución de métricas principales
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Distribución de Métricas por Estrategia", fontsize=16, fontweight="bold")

# Preparar datos para boxplots
ndcg_data = [df[f"{s}_ndcg"].dropna() for s in strategies]
precision_data = [df[f"{s}_precision"].dropna() for s in strategies]
recall_data = [df[f"{s}_recall"].dropna() for s in strategies]
f1_data = [df[f"{s}_f1"].dropna() for s in strategies]

# NDCG
axes[0, 0].boxplot(ndcg_data, labels=strategies)
axes[0, 0].set_title("NDCG@9", fontweight="bold")
axes[0, 0].set_ylabel("NDCG")
axes[0, 0].tick_params(axis="x", rotation=45)
axes[0, 0].grid(axis="y", alpha=0.3)

# Precision
axes[0, 1].boxplot(precision_data, labels=strategies)
axes[0, 1].set_title("Precision@9", fontweight="bold")
axes[0, 1].set_ylabel("Precision")
axes[0, 1].tick_params(axis="x", rotation=45)
axes[0, 1].grid(axis="y", alpha=0.3)

# Recall
axes[1, 0].boxplot(recall_data, labels=strategies)
axes[1, 0].set_title("Recall@9", fontweight="bold")
axes[1, 0].set_ylabel("Recall")
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].grid(axis="y", alpha=0.3)

# F1
axes[1, 1].boxplot(f1_data, labels=strategies)
axes[1, 1].set_title("F1@9", fontweight="bold")
axes[1, 1].set_ylabel("F1 Score")
axes[1, 1].tick_params(axis="x", rotation=45)
axes[1, 1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Ranking de estrategias por métrica
print("=" * 80)
print("RANKING DE ESTRATEGIAS POR MÉTRICA")
print("=" * 80)

for metric in [
    "ndcg",
    "precision",
    "recall",
    "f1",
    "mrr",
    "genre_diversity",
    "artist_diversity",
    "novelty",
]:
    print(f"\n{metric.upper()}:")
    print("-" * 40)
    ranked = summary_df[metric].sort_values(ascending=False)
    for i, (strategy, value) in enumerate(ranked.items(), 1):
        print(f"  {i}. {strategy:15s}: {value:.4f}")

In [ ]:
# Análisis de correlación entre métricas (para estrategias principales)
correlation_metrics = [
    "ndcg",
    "precision",
    "recall",
    "f1",
    "mrr",
    "genre_diversity",
    "artist_diversity",
    "novelty",
]
corr_data = []

for strategy in ["hybrid", "advanced", "pairs", "content"]:
    row = {}
    for metric in correlation_metrics:
        col = f"{strategy}_{metric}"
        if col in df.columns:
            row[metric] = df[col].mean()
    corr_data.append(row)

corr_df = pd.DataFrame(corr_data, index=["hybrid", "advanced", "pairs", "content"])

fig, ax = plt.subplots(figsize=(10, 8))
correlation_matrix = corr_df.corr()
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlación"},
    ax=ax,
)
ax.set_title(
    "Correlación entre Métricas (Estrategias principales)", fontsize=14, fontweight="bold", pad=20
)
plt.tight_layout()
plt.show()

In [ ]:
# Análisis por tamaño de holdout
df["holdout_category"] = pd.cut(
    df["holdout_size"],
    bins=[0, 20, 50, 100, 200, float("inf")],
    labels=[
        "Muy pequeño (≤20)",
        "Pequeño (21-50)",
        "Mediano (51-100)",
        "Grande (101-200)",
        "Muy grande (>200)",
    ],
)

print("=" * 80)
print("ANÁLISIS POR TAMAÑO DE HOLDOUT")
print("=" * 80)

for category in df["holdout_category"].cat.categories:
    subset = df[df["holdout_category"] == category]
    if len(subset) > 0:
        print(f"\n{category} ({len(subset)} usuarios):")
        print("-" * 40)
        for strategy in ["hybrid", "advanced", "pairs", "content"]:
            ndcg_col = f"{strategy}_ndcg"
            if ndcg_col in subset.columns:
                mean_ndcg = subset[ndcg_col].mean()
                print(f"  {strategy:15s}: NDCG promedio = {mean_ndcg:.4f}")

In [ ]:
# Visualización 5: Trade-off entre relevancia y diversidad
fig, ax = plt.subplots(figsize=(12, 8))

for strategy in strategies:
    ndcg_col = f"{strategy}_ndcg"
    div_col = f"{strategy}_genre_diversity"

    if ndcg_col in df.columns and div_col in df.columns:
        mean_ndcg = df[ndcg_col].mean()
        mean_div = df[div_col].mean()
        ax.scatter(mean_ndcg, mean_div, s=200, alpha=0.7, label=strategy)
        ax.annotate(
            strategy, (mean_ndcg, mean_div), xytext=(5, 5), textcoords="offset points", fontsize=9
        )

ax.set_xlabel("NDCG@9 (Relevancia)", fontweight="bold", fontsize=12)
ax.set_ylabel("Diversidad de Géneros", fontweight="bold", fontsize=12)
ax.set_title("Trade-off: Relevancia vs Diversidad", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="best", framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 6: Trade-off entre relevancia y novedad
fig, ax = plt.subplots(figsize=(12, 8))

for strategy in strategies:
    ndcg_col = f"{strategy}_ndcg"
    nov_col = f"{strategy}_novelty"

    if ndcg_col in df.columns and nov_col in df.columns:
        mean_ndcg = df[ndcg_col].mean()
        mean_nov = df[nov_col].mean()
        ax.scatter(mean_ndcg, mean_nov, s=200, alpha=0.7, label=strategy)
        ax.annotate(
            strategy, (mean_ndcg, mean_nov), xytext=(5, 5), textcoords="offset points", fontsize=9
        )

ax.set_xlabel("NDCG@9 (Relevancia)", fontweight="bold", fontsize=12)
ax.set_ylabel("Novedad", fontweight="bold", fontsize=12)
ax.set_title("Trade-off: Relevancia vs Novedad", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="best", framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusiones y Recomendaciones

### Resumen de Hallazgos:

1. **Mejor relevancia (NDCG)**: 
   - Estrategias con mejor NDCG promedio

2. **Mejor precisión**: 
   - Estrategias que mejor recuperan items relevantes

3. **Mejor diversidad**: 
   - Estrategias que ofrecen más variedad en géneros/artistas

4. **Mejor novedad**: 
   - Estrategias que descubren items menos populares

5. **Trade-offs identificados**:
   - Relación entre relevancia y diversidad
   - Relación entre relevancia y novedad

### Recomendaciones:

- **Para usuarios con historial corto**: 
- **Para usuarios con historial largo**: 
- **Para maximizar descubrimiento**: 
- **Para maximizar relevancia**:


In [ ]:
# Guardar resumen en CSV
summary_df.to_csv("../resultados_evaluacion_resumen.csv")
print("✅ Resumen guardado en: resultados_evaluacion_resumen.csv")

# Estadísticas adicionales
print("\n" + "=" * 80)
print("ESTADÍSTICAS ADICIONALES")
print("=" * 80)
print(f"\nTotal de usuarios evaluados: {len(df)}")
print(f"Tamaño promedio de holdout: {df['holdout_size'].mean():.1f}")
print(f"Tamaño mínimo de holdout: {df['holdout_size'].min()}")
print(f"Tamaño máximo de holdout: {df['holdout_size'].max()}")

# Usuarios con mejor rendimiento por estrategia
print("\n" + "=" * 80)
print("TOP 5 USUARIOS POR ESTRATEGIA (NDCG)")
print("=" * 80)
for strategy in ["hybrid", "advanced", "pairs", "content"]:
    ndcg_col = f"{strategy}_ndcg"
    if ndcg_col in df.columns:
        top_users = df.nlargest(5, ndcg_col)[["user_id", "holdout_size", ndcg_col]]
        print(f"\n{strategy.upper()}:")
        print(top_users.to_string(index=False))